## Correlation

**Correlation** is used to _test relationships between quantitative variables_

Some examples of data that have a high correlation:

1. Your caloric intake and your weight
1. The amount of time your study and your GPA

**Negative correlation** means the two variables move in opposite directions — as one goes up, the other tends to go down.

Correlations are useful because we can find out what relationship variables have, we can make predictions about future behavior.


## Activity: Obtain the correlation among all features of iris dataset

1. Review the iris dataset. What are the features?
1. Eliminate two columns `['Id', 'Species']`
1. Compute the correlation among all features.
    1. **Hint**: Use `df.corr()`
1. Plot the correlation as a heatmap in Seaborn -> `sns.heatmap`
1. Write a function that computes the correlation (Pearson formula)
    1. **Hint**: https://en.wikipedia.org/wiki/Pearson_correlation_coefficient
1. Compare your answer with `scipy.stats.pearsonr` for any given two features

<img src="./Images/iris_vis.jpg">


In [ ]:
import pandas as pd
import numpy as np
import scipy.stats
import seaborn as sns

df = pd.read_csv('Datasets/Iris.csv')
df = df.drop(columns=['Id', 'Species'])
sns.heatmap(df.corr(), annot=True)

def pearson_corr(x, y):
    x_mean = np.mean(x)
    y_mean = np.mean(y)
    num = [(i - x_mean)*(j - y_mean) for i,j in zip(x,y)]
    den_1 = [(i - x_mean)**2 for i in x]
    den_2 = [(j - y_mean)**2 for j in y]
    correlation_x_y = np.sum(num)/np.sqrt(np.sum(den_1))/np.sqrt(np.sum(den_2))
    return correlation_x_y

print(pearson_corr(df['SepalLengthCm'], df['PetalLengthCm']))
print(scipy.stats.pearsonr(df['SepalLengthCm'], df['PetalLengthCm']))


## Statistical Analysis: GMM and KDE

We can approximate the histogram of a dataset with a combination of Gaussian (Normal) distribution functions:

    Gaussian Mixture Model (GMM)
    
    Kernel Density Estimation (KDE)


## What is Gaussian Mixture Model (GMM)?

GMM is a probabilistic model for representing normally distributed subpopulations within an overall population

<img src="Images/gmm_fig.png" width="300">

$p(x) = \sum_{i = 1}^{K} w_i \ \mathcal{N}(x \ | \ \mu_i,\ \sigma_i)$

$\sum_{i=1}^{K} w_i = 1$

https://brilliant.org/wiki/gaussian-mixture-model/


## What is the goal for using GMM?

- We want to approximate the density (histogram) of any given data sample with a combination of Normal Distributions

- How many Normal Distributions we need is defined by us (2 or 3 or 4, ...)


## Activity: Fit a GMM to a given data sample

Task:

1. Generate the concatenation of the random variables as follows:

`x_1 = np.random.normal(-5, 1, 3000)`

`x_2 = np.random.normal(2, 3, 7000)`

`x = np.concatenate((x_1, x_2))`

2. Plot the histogram of `x`
3. Obtain the weights, mean and variances of each Gaussian

You will need to use the following in your solution:
```python
from sklearn import mixture

gmm = mixture.GaussianMixture(n_components=2)

gmm.fit(x.reshape(-1,1))
```


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import mixture

# Generate data samples and plot its histogram
x_1 = np.random.normal(-5, 1, 3000)
x_2 = np.random.normal(2, 3, 7000)
x = np.concatenate((x_1, x_2))
plt.hist(x, bins=20, density=1)
plt.show()

# Define a GMM model and obtain its parameters
gmm = mixture.GaussianMixture(n_components=2)
gmm.fit(x.reshape(-1,1))
print(gmm.means_)
print(gmm.covariances_)
print(gmm.weights_)


## The GMM has learned the probability density function of our data sample

Let's generate samples from the GMM:


In [ ]:
z = gmm.sample(10000)
plt.hist(z[0], bins=20, density=1)
plt.show()


## Question: Are the samples in z and x the same?

Not their samples, but we know that the _pdf of x and the pdf of z are the same_ — the GMM learned the shape of the distribution, not the exact data points.


## Kernel Density Estimation (KDE)

**Kernel density estimation (KDE)** is a non-parametric way to estimate the Probability Density Function (PDF) of a data sample. In other words _the goal of KDE is to find the PDF for a given data sample._

We can use the below formula to approximate the PDF of the dataset:

$p(x) = \frac{1}{Nh}\sum_{i = 1}^{N} \ K(\frac{x - x_i}{h})$

where $h$ is a bandwidth and $N$ is the number of data points


## Activity: Apply KDE on a given data sample

**Task**: Apply KDE on the previously generated sample data `x`

**Hint**: use

`kde = KernelDensity(kernel='gaussian', bandwidth=0.6)`


In [ ]:
from sklearn.neighbors import KernelDensity

kde = KernelDensity(kernel='gaussian', bandwidth=0.6)
kde.fit(x.reshape(-1,1))

s = np.linspace(np.min(x), np.max(x))
log_pdf = kde.score_samples(s.reshape(-1,1))
plt.plot(s, np.exp(log_pdf))


## The KDE has learned the probability density function of our data sample

Let's generate samples from the KDE:


In [ ]:
m = kde.sample(10000)
plt.hist(m, bins=20, density=1)
plt.show()


## KDE can learn handwritten digits distribution and generate new digits

http://scikit-learn.org/stable/auto_examples/neighbors/plot_digits_kde_sampling.html
